# CBIS-DDSM → DeepProbLog

This notebook is the full pipeline: **prepare the CSVs**, **load mammogram JPEGs**, **augment the training images**, **search hyperparameters**, **train DeepProbLog**, then **evaluate and show images**.

## Goal of DeepProbLog

A plain CNN maps pixels straight to `malignant` / `benign`. You get a score, but not a reason.

DeepProbLog is **neuro-symbolic**:

1. Small neural nets look at the image and predict **concepts** a radiologist would name: lesion kind (mass vs calcification), mass shape, mass margin, and calcification type.
2. A **Prolog program** turns those uncertain concept probabilities into a diagnosis with explicit rules, for example:
   - a **mass** is called malignant if the shape is irregular or the margin is spiculated
   - a **calcification** is called malignant if the type is suspicious
3. Training updates the neural nets so both the concepts and the final `malignant(X)` query match the labels.

The point is not only accuracy. It is to keep the decision in a form you can inspect: if `P(malignant)` is high, the traces show *which* concept fired.

## Steps in this notebook

| Step | What happens |
|---|---|
| 1–3 | Install, import, set config |
| 4–6 | Load CSVs, merge mass+calc, remove patient leakage, save splits |
| 7 | Map CSV text to Prolog concept symbols |
| 8 | Load JPEGs and show train / val / test images |
| 8b | **Augment training images only** (flips, small rotations) to grow the train set |
| 9 | Define the concept CNNs and DeepProbLog datasets |
| 10 | **Grid-search** learning rate and batch size on validation F1, then train the final model |
| 11 | Metrics, reasoning traces, predicted test images |
| 12 | Save weights and metrics |

## Kaggle setup

- Attach dataset: `cbis-ddsm-breast-cancer-image-dataset`
- Accelerator: GPU T4 (P100 is not compatible with this PyTorch build)
- First check: `SMOKE_TEST = True` (tiny sample, fast wiring test)
- Full run: `SMOKE_TEST = False`
- `RUN_GRID_SEARCH = True` tries a few `(lr, batch)` pairs with fewer epochs, then retrains with the best pair

## How to read the log

- `DEVICE: cuda` means the GPU is in use.
- `loaded N tensors (0 synthetic)` means real JPEGs were found. If synthetic is not 0, the images are placeholders and the run is only a smoke test.

The Prolog rules are a computational working program, not official BI-RADS criteria.


## 1) Install packages

**What:** install DeepProbLog, scikit-learn, pandas, and Pillow in this kernel.

**Why:** Kaggle runtimes do not always include DeepProbLog. This cell is safe to re-run.


In [ ]:
%pip install -q deepproblog scikit-learn pandas pillow

## 2) Imports

**What:** load libraries for tables, images, PyTorch, and plots.

**Why:** later cells need pandas for CSVs, matplotlib to show mammograms, and torch for the concept networks.


In [ ]:
import json
import os
import random
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torchvision import transforms

print("pandas:", pd.__version__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 3) Config

**What:** one settings cell for the whole notebook.

**Why:** change smoke-test size, training length, augmentation, and the hyperparameter grid here.

- `SMOKE_TEST=True` — tiny wiring test (few patients, 1 epoch).
- `SMOKE_TEST=False` — full split and 15 epochs.
- `N_AUGMENT` — extra random views per **training** image only. Validation and test stay clean.
- `BALANCE_CLASSES=True` — malignant training rows get extra augmented copies until both classes are equal in size.
- `THRESHOLD_MODE` / `TARGET_SENSITIVITY` — how the decision threshold on `P(malignant)` is chosen on validation. `0.5` under-calls malignant; `target_sensitivity` lowers it until validation recall reaches the target.
- `RUN_GRID_SEARCH=True` — try each `(lr, batch_size)` for `SEARCH_EPOCHS`, pick the best validation macro F1, then retrain with `EPOCHS`.
- `USE_SYNTHETIC_IF_MISSING=False` — missing JPEGs must raise an error.
- `FORCE_CPU=False` — use T4 if the CUDA probe succeeds.


In [ ]:
# ===================== ONE CONFIG FOR EVERYTHING =====================
SMOKE_TEST = False                 # True = tiny wiring test; False = full run
SMOKE_N_PATIENTS = 20             # patients per official split (data prep)
SMOKE_N_SAMPLES = 24              # max rows used for DeepProbLog tensors
VAL_SIZE = 0.15
RANDOM_SEED = 42

EPOCHS = 1 if SMOKE_TEST else 15         # full run: 15 epochs
BATCH_SIZE = 1 if SMOKE_TEST else 2      # default if grid search is off; drop to 2 on OOM
IMAGE_SIZE = 64 if SMOKE_TEST else 128   # 128 sees more of the lesion than 64
LR = 0.0003                                # default if grid search is off
USE_SYNTHETIC_IF_MISSING = False  # missing JPEG must fail; do not invent images
FORCE_CPU = False                 # use GPU if the probe succeeds (T4). P100 still falls back to CPU.
SHOW_N_IMAGES = 8                 # pictures to draw in each image grid

# Train-only augmentation: each original train image is kept, plus N_AUGMENT random copies.
N_AUGMENT = 1 if SMOKE_TEST else 2

# Class balance. Malignant is ~41% of train, so at 0.5 the model under-calls malignant.
# With BALANCE_CLASSES=True the malignant rows get extra augmented copies until both
# classes have the same number of training images. Validation and test are untouched.
BALANCE_CLASSES = True

# Decision threshold on P(malignant). The threshold is chosen on VALIDATION only and
# then applied unchanged to TEST, so the test numbers stay honest.
#   "fixed"              -> use DECISION_THRESHOLD as is
#   "target_sensitivity" -> lowest threshold whose validation sensitivity >= TARGET_SENSITIVITY
#   "youden"             -> maximize sensitivity + specificity - 1
#   "f1_macro"           -> maximize macro F1
THRESHOLD_MODE = "target_sensitivity"
TARGET_SENSITIVITY = 0.80
DECISION_THRESHOLD = 0.5

# Small grid on validation macro F1. Keep it small: DeepProbLog exact inference is slow.
RUN_GRID_SEARCH = True
SEARCH_EPOCHS = 1 if SMOKE_TEST else 3
GRID_LEARNING_RATES = [3e-4, 1e-3] if SMOKE_TEST else [3e-4, 1e-3, 3e-3]
GRID_BATCH_SIZES = [2] if SMOKE_TEST else [2, 4]

DATASET_HANDLE = "awsaf49/cbis-ddsm-breast-cancer-image-dataset"
KAGGLE_INPUT = Path("/kaggle/input/cbis-ddsm-breast-cancer-image-dataset")
KAGGLE_WORKING = Path("/kaggle/working")
LOCAL_DATA = Path("../data")

CSV_FILENAMES = {
    "mass_train": "mass_case_description_train_set.csv",
    "mass_test": "mass_case_description_test_set.csv",
    "calc_train": "calc_case_description_train_set.csv",
    "calc_test": "calc_case_description_test_set.csv",
}

PROGRAM_FALLBACK = """
nn(kind_net, [X], Y, [mass, calcification]) :: lesion_kind(X, Y).
nn(shape_net, [X], Y, [regular, irregular, unknown]) :: shape(X, Y).
nn(margin_net, [X], Y, [circumscribed, spiculated, other, unknown]) :: margin(X, Y).
nn(calc_net, [X], Y, [absent, typically_benign, suspicious]) :: calc(X, Y).

malignant(X) :- lesion_kind(X, mass), shape(X, irregular).
malignant(X) :- lesion_kind(X, mass), margin(X, spiculated).
malignant(X) :- lesion_kind(X, calcification), calc(X, suspicious).
""".strip()

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

def pick_device(force_cpu=False):
    """Prefer CUDA, but fall back to CPU if the binary can't run on this GPU.
    Common on Kaggle when PyTorch wheels don't match the accelerator arch.
    """
    if force_cpu or not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        # Probe that kernels actually work on this GPU (not just "cuda available").
        x = torch.zeros(1, device="cuda")
        y = x + 1
        torch.cuda.synchronize()
        del x, y
        torch.cuda.empty_cache()
        return torch.device("cuda")
    except Exception as exc:
        print("CUDA probe failed → using CPU. Reason:", type(exc).__name__, str(exc)[:200])
        return torch.device("cpu")

DEVICE = pick_device(FORCE_CPU)
print("DEVICE:", DEVICE)
if DEVICE.type == "cpu" and torch.cuda.is_available():
    print("Note: CUDA is visible but unusable with this PyTorch build. DeepProbLog will run on CPU.")

if Path("/kaggle/working").is_dir():
    OUTPUT_DIR = Path("/kaggle/working/processed")
else:
    OUTPUT_DIR = Path("../notebooks/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if SMOKE_TEST:
    print("=" * 64)
    print("SMOKE_TEST=True — tiny wiring test on a small sample.")
    print("=" * 64)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

## 4) Find the four CBIS CSVs

**What:** search Kaggle input, then local `data/`, for the official mass and calc description files.

**Why:** the files are not always at a fixed path. Walking the tree (skipping `jpeg/`) keeps this cell working on Kaggle and on a laptop.


In [ ]:
def find_csv_paths(root, filenames):
    root = Path(root)
    found, remaining = {}, set(filenames.values())
    for dirpath, dirnames, files in os.walk(root):
        dirnames[:] = [d for d in dirnames if d.lower() != "jpeg"]
        for fname in files:
            if fname in remaining:
                key = next(k for k, v in filenames.items() if v == fname)
                found[key] = Path(dirpath) / fname
                remaining.discard(fname)
        if not remaining:
            break
    missing = [k for k in filenames if k not in found]
    if missing:
        raise FileNotFoundError(f"Missing {missing} under {root}")
    return found


def resolve_data_root():
    if KAGGLE_INPUT.is_dir():
        return KAGGLE_INPUT, "kaggle_input"
    # other common Kaggle mount
    for p in Path("/kaggle/input").glob("*cbis*") if Path("/kaggle/input").is_dir() else []:
        if p.is_dir():
            return p, "kaggle_input_scan"
    if LOCAL_DATA.is_dir() and all((LOCAL_DATA / fn).exists() for fn in CSV_FILENAMES.values()):
        return LOCAL_DATA, "local_data"
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(DATASET_HANDLE))
        return downloaded, "kagglehub_download"
    except Exception as exc:
        raise FileNotFoundError("No CBIS-DDSM data found") from exc


DATA_ROOT, DATA_SOURCE = resolve_data_root()
CSV_PATHS = find_csv_paths(DATA_ROOT, CSV_FILENAMES)
print(f"Data source: {DATA_SOURCE}\nRoot: {DATA_ROOT}")
for k, v in CSV_PATHS.items():
    print(f"  {k}: {v}")

## 5) Merge mass+calc, fix leakage, split by patient

**What:** concatenate mass and calcification tables, assign overlapping patients entirely to train, then cut a patient-disjoint validation set.

**Why:** after merging, some patients appear in both official train and test. A patient-level split is required so the same woman is never in train and test.


In [ ]:
STRING_COLUMNS = [
    "patient_id", "left or right breast", "image view", "abnormality type",
    "mass shape", "mass margins", "calc type", "calc distribution", "pathology",
    "image file path", "cropped image file path", "ROI mask file path",
]


def normalize_columns(df):
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    if "breast density" in out.columns:
        out = out.rename(columns={"breast density": "breast_density"})
    for col in STRING_COLUMNS:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip().replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return out


def add_meta(df, kind, official_split):
    out = df.copy()
    out["abnormality_kind"] = kind
    out["official_split"] = official_split
    return out


def sample_patients(df, n_patients, seed):
    patients = df["patient_id"].dropna().unique()
    if len(patients) <= n_patients:
        return df.copy()
    chosen = set(random.Random(seed).sample(list(patients), n_patients))
    return df[df["patient_id"].isin(chosen)].copy()


def resolve_cross_split_patient_leakage(train_df, test_df):
    """CBIS mass/calc official CSVs can place the same patient in train AND test
    (e.g. mass in train, calc in test). Keep ALL rows for those patients in train
    and drop them from test so splits stay patient-disjoint.
    """
    train_p = set(train_df["patient_id"].dropna())
    test_p = set(test_df["patient_id"].dropna())
    overlap = train_p & test_p
    if not overlap:
        print("No cross-split patient overlap.")
        return train_df, test_df

    print(f"Found {len(overlap)} patients in BOTH official train and test after merge.")
    print("  Assigning all their rows to TRAIN and removing them from TEST.")
    leaked_test_rows = test_df[test_df["patient_id"].isin(overlap)].copy()
    train_fixed = pd.concat([train_df, leaked_test_rows], ignore_index=True)
    test_fixed = test_df[~test_df["patient_id"].isin(overlap)].reset_index(drop=True)
    # drop duplicate rows if any exact duplicates appear
    train_fixed = train_fixed.drop_duplicates().reset_index(drop=True)
    return train_fixed, test_fixed


raw = {k: pd.read_csv(v) for k, v in CSV_PATHS.items()}
mass_train = add_meta(normalize_columns(raw["mass_train"]), "mass", "train")
mass_test = add_meta(normalize_columns(raw["mass_test"]), "mass", "test")
calc_train = add_meta(normalize_columns(raw["calc_train"]), "calcification", "train")
calc_test = add_meta(normalize_columns(raw["calc_test"]), "calcification", "test")

df_train_all = pd.concat([mass_train, calc_train], ignore_index=True)
df_test_all = pd.concat([mass_test, calc_test], ignore_index=True)
print("Official train:", len(df_train_all), "rows |", df_train_all["patient_id"].nunique(), "patients")
print("Official test: ", len(df_test_all), "rows |", df_test_all["patient_id"].nunique(), "patients")

df_train_all, df_test_all = resolve_cross_split_patient_leakage(df_train_all, df_test_all)
print("After leakage fix train:", len(df_train_all), "rows |", df_train_all["patient_id"].nunique(), "patients")
print("After leakage fix test: ", len(df_test_all), "rows |", df_test_all["patient_id"].nunique(), "patients")

if SMOKE_TEST:
    df_train_all = sample_patients(df_train_all, SMOKE_N_PATIENTS, RANDOM_SEED)
    df_test_all = sample_patients(df_test_all, SMOKE_N_PATIENTS, RANDOM_SEED + 1)
    print("After smoke sampling:", len(df_train_all), "train rows |", len(df_test_all), "test rows")

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_SEED)
tr_idx, va_idx = next(gss.split(df_train_all, groups=df_train_all["patient_id"]))
df_train = df_train_all.iloc[tr_idx].reset_index(drop=True)
df_val = df_train_all.iloc[va_idx].reset_index(drop=True)
df_test = df_test_all.reset_index(drop=True)

tr_p, va_p, te_p = set(df_train.patient_id), set(df_val.patient_id), set(df_test.patient_id)
print("overlap train∩val:", len(tr_p & va_p), "| train∩test:", len(tr_p & te_p), "| val∩test:", len(va_p & te_p))
assert tr_p.isdisjoint(va_p), "Train/val patient leakage"
assert tr_p.isdisjoint(te_p), "Train/test patient leakage"
assert va_p.isdisjoint(te_p), "Val/test patient leakage"
print("Leakage checks passed.")
for name, df in (("train", df_train), ("val", df_val), ("test", df_test)):
    print(f"{name:5s}: {len(df):>5} rows | {df['patient_id'].nunique():>4} patients")


## 6) Save the split tables

**What:** write `cbis_train.csv`, `cbis_val.csv`, `cbis_test.csv`, and a patient-to-split map.

**Why:** so you can inspect exactly which cases this run used.


In [ ]:
# Save processed artifacts (same session will also train on these frames)
patient_splits = []
for split_name, frame in (("train", df_train), ("val", df_val), ("test", df_test)):
    for pid in sorted(frame["patient_id"].dropna().unique()):
        patient_splits.append({"patient_id": pid, "split": split_name})
pd.DataFrame(patient_splits).to_csv(OUTPUT_DIR / "cbis_patient_splits.csv", index=False)
df_train.to_csv(OUTPUT_DIR / "cbis_train.csv", index=False)
df_val.to_csv(OUTPUT_DIR / "cbis_val.csv", index=False)
df_test.to_csv(OUTPUT_DIR / "cbis_test.csv", index=False)

meta = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "smoke_test": SMOKE_TEST,
    "smoke_n_patients": SMOKE_N_PATIENTS if SMOKE_TEST else None,
    "val_size": VAL_SIZE,
    "random_seed": RANDOM_SEED,
    "data_source": DATA_SOURCE,
    "data_root": str(DATA_ROOT),
    "counts": {
        "train_rows": len(df_train),
        "val_rows": len(df_val),
        "test_rows": len(df_test),
        "train_patients": df_train["patient_id"].nunique(),
        "val_patients": df_val["patient_id"].nunique(),
        "test_patients": df_test["patient_id"].nunique(),
    },
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))
print("Saved prep outputs to", OUTPUT_DIR.resolve())

## 7) Map CSV descriptors to concepts

**What:** convert `mass shape`, `mass margins`, `calc type`, and `abnormality_kind` into the symbols used in the Prolog program.

**Why:** DeepProbLog queries `shape(X, irregular)`, not the original CBIS strings. Any calcification is **not** treated as malignant; only suspicious types map to `suspicious`.


In [ ]:
def tokenize_descriptor(value):
    if pd.isna(value):
        return []
    text = str(value).strip().upper()
    if text in {"", "NAN", "NONE"}:
        return []
    for sep in ("-", "/", "_", " "):
        text = text.replace(sep, " ")
    return [t for t in text.split() if t]


def map_shape(value):
    tokens = tokenize_descriptor(value)
    if not tokens:
        return "unknown"
    if any(t in {"IRREGULAR", "ARCHITECTURAL", "DISTORTION"} for t in tokens):
        return "irregular"
    if any(t in {"ROUND", "OVAL", "LOBULATED"} for t in tokens):
        return "regular"
    return "unknown"


def map_margin(value):
    tokens = tokenize_descriptor(value)
    if not tokens:
        return "unknown"
    if "SPICULATED" in tokens:
        return "spiculated"
    if "CIRCUMSCRIBED" in tokens:
        return "circumscribed"
    return "other"


def map_calc(row):
    # Any calcification present is NOT a malignancy label. Most CBIS calc cases are benign.
    if str(row.get("abnormality_kind", "")).lower() != "calcification":
        return "absent"
    tokens = tokenize_descriptor(row.get("calc type"))
    if any(t in {"PLEOMORPHIC", "AMORPHOUS"} for t in tokens) or (
        "FINE" in tokens and "LINEAR" in tokens
    ) or "BRANCHING" in tokens:
        return "suspicious"
    return "typically_benign"


def add_concepts(df):
    out = df.copy()
    out["shape_concept"] = out["mass shape"].map(map_shape) if "mass shape" in out.columns else "unknown"
    out["margin_concept"] = out["mass margins"].map(map_margin) if "mass margins" in out.columns else "unknown"
    out["calc_concept"] = out.apply(map_calc, axis=1)
    out["kind_concept"] = out["abnormality_kind"].map(
        lambda x: "calcification" if str(x).strip().lower() == "calcification" else "mass"
    )
    out["malignant"] = out["pathology"].map(lambda x: 1.0 if str(x).strip().upper() == "MALIGNANT" else 0.0)
    return out


df_train = add_concepts(df_train)
df_val = add_concepts(df_val)
df_test = add_concepts(df_test)
display(df_train[["patient_id", "abnormality_kind", "shape_concept", "margin_concept", "calc_concept", "malignant"]].sample(10))
print("calc_concept counts:\n", df_train["calc_concept"].value_counts())
print("malignant rate train/val/test:",
      float(df_train["malignant"].mean()),
      float(df_val["malignant"].mean()),
      float(df_test["malignant"].mean()))

## 8) Load JPEGs, show images, augment and balance training data

**What:** resolve each CSV path to a JPEG folder (by DICOM UID), prefer a gray-level photo over a binary mask, materialize tensors, draw image grids, then **augment only the training set**.

**Why:** you need to see the actual mammogram crops (`synthetic` must be 0). Extra flipped/rotated copies of train images give the concept nets more variety without touching validation or test, so the test numbers stay honest.

**Class balance:** the train split is about 41% malignant. A model trained on that mix learns to lean benign, which is the wrong direction for a screening task. With `BALANCE_CLASSES=True`, benign rows get `N_AUGMENT` copies and malignant rows get as many copies as needed so both classes end up with the same count. Augmentation is geometric only (flip, small rotation, small scale/shift), so concept labels such as `irregular` or `spiculated` stay valid on the copies.


In [ ]:
import os
import time
import concurrent.futures
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

NUM_WORKERS = min(8, (os.cpu_count() or 4) * 2)  # I/O-bound → oversubscribe cores a bit
THUMBNAIL_FOR_SCORE = 256  # small copy used only to pick photo vs mask, not for training


def find_jpeg_root():
    candidates = [DATA_ROOT / "jpeg", KAGGLE_INPUT / "jpeg", LOCAL_DATA / "jpeg"]
    if Path("/kaggle/input").is_dir():
        candidates.extend(p for p in Path("/kaggle/input").rglob("jpeg") if p.is_dir())
    for p in candidates:
        if p.is_dir():
            return p
    return None


JPEG_ROOT = find_jpeg_root()
print("JPEG root:", JPEG_ROOT)
print("Tensors will be placed on:", DEVICE)
print(f"Parallel image loading workers: {NUM_WORKERS}")

tfm = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])


def clean_relpath(value):
    if pd.isna(value):
        return None
    text = str(value).strip().strip('"').split("\n")[0].strip()
    return text or None


_JPEG_INDEX = None


def jpeg_folder_index():
    """Map folder names that actually contain images to their paths.

    The awsaf49 jpeg tree is not the CSV DICOM path. Images usually live in a
    folder named by the series UID (3rd path component), as 1-1.jpg / 1-2.jpg.
    """
    global _JPEG_INDEX
    if _JPEG_INDEX is not None:
        return _JPEG_INDEX
    index = {}
    if JPEG_ROOT is None or not JPEG_ROOT.is_dir():
        _JPEG_INDEX = index
        return index
    t0 = time.perf_counter()
    for dirpath, _dirnames, filenames in os.walk(JPEG_ROOT):
        imgs = [f for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if not imgs:
            continue
        folder = Path(dirpath)
        index[folder.name] = folder
    _JPEG_INDEX = index
    dt = time.perf_counter() - t0
    print(f"Indexed {len(index)} jpeg folders in {dt:.1f}s")
    print("example jpeg folders:", list(index)[:5])
    return index


def image_photo_score(path):
    """Higher score = more like a mammogram crop, less like a binary ROI mask.

    Scored on a small thumbnail: we only need the ranking between candidate
    files in the same folder, not exact statistics, so this is much cheaper
    than scoring the full-resolution image.
    """
    with Image.open(path) as im:
        im = im.convert("L")
        im.thumbnail((THUMBNAIL_FOR_SCORE, THUMBNAIL_FOR_SCORE))
        arr = np.asarray(im, dtype=np.uint8)
    n_unique = int(len(np.unique(arr)))
    mid_gray = float(((arr > 15) & (arr < 240)).mean())
    return (n_unique, mid_gray, float(arr.std()))


def pick_image(folder, prefer_crop=True):
    folder = Path(folder)
    jpgs = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.jpeg")) + sorted(folder.glob("*.png"))
    if not jpgs:
        jpgs = sorted(folder.rglob("*.jpg")) + sorted(folder.rglob("*.jpeg")) + sorted(folder.rglob("*.png"))
    if not jpgs:
        return None
    if len(jpgs) == 1:
        return jpgs[0]
    # CBIS crop/mask folders contain 1-1.jpg and 1-2.jpg, and the CSV columns
    # are often swapped. Prefer the file with many gray levels (photo), not a mask.
    return max(jpgs, key=image_photo_score)


def resolve_jpeg(row):
    if JPEG_ROOT is None:
        return None
    index = jpeg_folder_index()
    for col in ("cropped image file path", "image file path"):
        rel = clean_relpath(row.get(col))
        if not rel:
            continue
        parts = [p for p in rel.replace("\\", "/").split("/") if p]
        if not parts:
            continue
        prefer_crop = col.startswith("cropped")
        exact = JPEG_ROOT / rel.replace(".dcm", ".jpg")
        if exact.is_file():
            return exact
        keys = []
        # Do not use Path.stem here. DICOM UIDs contain dots, so stem()
        # would chop the series id and never match the jpeg folder name.
        for part in parts:
            key = part[:-4] if part.lower().endswith(".dcm") else part
            if key:
                keys.append(key)
        seen = set()
        for key in keys:
            if not key or key in seen:
                continue
            seen.add(key)
            folder = index.get(key)
            if folder is None and (JPEG_ROOT / key).is_dir():
                folder = JPEG_ROOT / key
            if folder is None:
                continue
            img = pick_image(folder, prefer_crop=prefer_crop)
            if img is not None:
                return img
    return None


def _to_device(t):
    """Move tensor to DEVICE; if CUDA fails mid-run, switch globally to CPU."""
    global DEVICE
    try:
        return t.to(DEVICE)
    except Exception as exc:
        print("Device move failed (", type(exc).__name__, ") → forcing CPU for the rest of the run.")
        DEVICE = torch.device("cpu")
        return t.to(DEVICE)


def concept_placeholder(row):
    """Used only when a JPEG is missing. These placeholders are for a smoke test, not a real-image run."""
    shape_c = row.get("shape_concept", "unknown")
    margin_c = row.get("margin_concept", "unknown")
    calc_c = row.get("calc_concept", "absent")
    t = torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE)
    third = IMAGE_SIZE // 3
    t[0, :third, :] = {"irregular": 1.0, "regular": 0.55, "unknown": 0.15}.get(shape_c, 0.15)
    t[0, third:2 * third, :] = {"spiculated": 1.0, "circumscribed": 0.55, "other": 0.35, "unknown": 0.15}.get(margin_c, 0.15)
    t[0, 2 * third:, :] = {"suspicious": 1.0, "typically_benign": 0.55, "absent": 0.15}.get(calc_c, 0.15)
    return t


def load_tensor(row):
    path = resolve_jpeg(row)
    if path is not None and path.is_file():
        with Image.open(path) as img:
            tensor = _to_device(tfm(img.convert("L")))
            return tensor, str(path), False
    if not USE_SYNTHETIC_IF_MISSING:
        rel = clean_relpath(row.get("cropped image file path")) or clean_relpath(row.get("image file path"))
        raise FileNotFoundError(
            f"No JPEG for patient {row.get('patient_id')}. csv_path={rel!r}. "
            f"JPEG_ROOT={JPEG_ROOT}. Check the 'Indexed N jpeg folders' line above."
        )
    return _to_device(concept_placeholder(row)), None, True


def materialize_split(df, n_limit=None, split_name="split"):
    if n_limit is not None:
        df = df.head(n_limit).copy()
    rows = list(df.iterrows())
    n = len(rows)
    tensors = [None] * n
    paths = [None] * n
    synths = [False] * n
    t_start = time.perf_counter()

    def _worker(item):
        i, (_, row) = item
        tensor, path, synthetic = load_tensor(row)
        return i, tensor, path, synthetic

    n_synth = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futures = ex.map(_worker, enumerate(rows))
        for i, tensor, path, synthetic in tqdm(futures, total=n, desc=f"Materializing {split_name}", unit="img"):
            tensors[i] = tensor
            paths[i] = path
            synths[i] = synthetic
            n_synth += int(synthetic)

    elapsed = time.perf_counter() - t_start
    avg_ms = (elapsed / max(n, 1)) * 1000
    print(f"  loaded {n} tensors ({n_synth} synthetic) on {DEVICE} "
          f"in {elapsed:.1f}s (avg {avg_ms:.0f} ms/image, {NUM_WORKERS} workers)")

    records = []
    for (_, row), path, synthetic in zip(rows, paths, synths):
        rec = row.to_dict()
        rec["resolved_path"] = path
        rec["synthetic"] = synthetic
        records.append(rec)
    return tensors, pd.DataFrame(records)


# Build the folder index once, up front, before the thread pool starts —
# avoids every worker thread racing to build it redundantly on first access.
jpeg_folder_index()

limit = SMOKE_N_SAMPLES if SMOKE_TEST else None
t_all_start = time.perf_counter()
print("Materializing train"); train_tensors, df_train = materialize_split(df_train, limit, split_name="train")
print("Materializing val");   val_tensors, df_val = materialize_split(df_val, limit, split_name="val")
print("Materializing test");  test_tensors, df_test = materialize_split(df_test, limit, split_name="test")
print(f"Total materialization time: {time.perf_counter() - t_all_start:.1f}s")

if df_train["synthetic"].all():
    print("WARNING: all train images are synthetic placeholders — check JPEG_ROOT on Kaggle.")
    print("This is a smoke test only until real JPEGs load (synthetic count must be 0).")
elif df_train["synthetic"].any():
    print("WARNING: some train images are synthetic placeholders.")


def tensor_to_numpy(t):
    x = t.detach().cpu() if hasattr(t, "detach") else t
    if hasattr(x, "numpy"):
        x = x.numpy()
    x = np.squeeze(x)
    return x


def show_image_grid(title, df, tensors, n=None):
    n = min(int(n if n is not None else SHOW_N_IMAGES), len(df), len(tensors))
    if n == 0:
        print(title, ": no images")
        return
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(13, 3.2 * rows))
    axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        ax = axes[i]
        ax.imshow(tensor_to_numpy(tensors[i]), cmap="gray")
        row = df.iloc[i]
        synth = " [placeholder]" if bool(row.get("synthetic", False)) else ""
        ax.set_title(
            f"{row['patient_id']}{synth}\n{row['pathology']}\n{row['abnormality_kind']}",
            fontsize=8,
        )
        ax.axis("off")
    for j in range(n, len(axes)):
        axes[j].axis("off")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.show()


print("\nShowing loaded images (real JPEG if synthetic=0):")
show_image_grid("Train images", df_train, train_tensors)
show_image_grid("Validation images", df_val, val_tensors)
show_image_grid("Test images", df_test, test_tensors)


# ---- train-only augmentation (val/test stay unchanged) ----
aug_tfm = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12, fill=0),
    transforms.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.92, 1.08), fill=0),
])


def augment_one(tensor, rng_seed):
    x = tensor.detach().cpu() if hasattr(tensor, "detach") else torch.as_tensor(tensor)
    x = x.to(dtype=torch.float32)
    if x.dim() == 2:
        x = x.unsqueeze(0)
    g = torch.Generator()
    g.manual_seed(int(rng_seed) % (2**31 - 1))
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(int(rng_seed) % (2**31 - 1))
        x = aug_tfm(x)
    return x


df_train_orig = df_train.reset_index(drop=True)
train_tensors_orig = list(train_tensors)
n_mal = int((df_train_orig["malignant"] == 1).sum())
n_ben = int((df_train_orig["malignant"] == 0).sum())
print(f"Train before augment: {len(df_train_orig)} images | malignant={n_mal} benign={n_ben} "
      f"(malignant rate {n_mal / max(len(df_train_orig), 1):.3f})")


def augmented_copies(indices, n_copies_total, seed_base, pbar=None):
    """Produce n_copies_total augmented copies by cycling through `indices`.

    Used for class balancing: the minority class needs more copies than the
    majority class, so we keep cycling (with a fresh shuffle each pass) until
    the requested count is reached. Every copy gets its own RNG seed.
    """
    rows, tensors = [], []
    if n_copies_total <= 0 or len(indices) == 0:
        return rows, tensors
    rng = random.Random(seed_base)
    order = list(indices)
    k = 0
    while len(tensors) < n_copies_total:
        rng.shuffle(order)
        for i in order:
            if len(tensors) >= n_copies_total:
                break
            rows.append(df_train_orig.iloc[i].to_dict())
            tensors.append(augment_one(train_tensors_orig[i], seed_base + 7919 * k + i))
            k += 1
            if pbar is not None:
                pbar.update(1)
    return rows, tensors


if N_AUGMENT > 0 or BALANCE_CLASSES:
    ben_idx = df_train_orig.index[df_train_orig["malignant"] == 0].tolist()
    mal_idx = df_train_orig.index[df_train_orig["malignant"] == 1].tolist()
    n_ben_extra = N_AUGMENT * n_ben
    n_mal_extra = N_AUGMENT * n_mal
    if BALANCE_CLASSES:
        # Both classes end up at the size of the larger augmented class.
        target_per_class = max(n_ben + n_ben_extra, n_mal + n_mal_extra)
        n_ben_extra = target_per_class - n_ben
        n_mal_extra = target_per_class - n_mal
        print(f"BALANCE_CLASSES=True -> target {target_per_class} images per class "
              f"(+{n_ben_extra} benign copies, +{n_mal_extra} malignant copies)")

    t_aug_start = time.perf_counter()
    pbar = tqdm(total=n_ben_extra + n_mal_extra, desc="Augmenting train", unit="img")
    ben_rows, ben_tensors = augmented_copies(ben_idx, n_ben_extra, RANDOM_SEED + 1000, pbar)
    mal_rows, mal_tensors = augmented_copies(mal_idx, n_mal_extra, RANDOM_SEED + 2000, pbar)
    pbar.close()
    aug_elapsed = time.perf_counter() - t_aug_start

    extra_rows = ben_rows + mal_rows
    extra_tensors = ben_tensors + mal_tensors
    df_train = pd.concat([df_train_orig, pd.DataFrame(extra_rows)], ignore_index=True)
    train_tensors = [_to_device(t) if torch.is_tensor(t) else t for t in (train_tensors_orig + extra_tensors)]

    n_mal_after = int((df_train["malignant"] == 1).sum())
    n_ben_after = int((df_train["malignant"] == 0).sum())
    print(f"Train after augment: {len(df_train)} images | malignant={n_mal_after} benign={n_ben_after} "
          f"(malignant rate {n_mal_after / max(len(df_train), 1):.3f}) in {aug_elapsed:.1f}s")
    if mal_tensors:
        show_image_grid("Augmented MALIGNANT train copies (examples)", pd.DataFrame(mal_rows), mal_tensors,
                        n=min(SHOW_N_IMAGES, len(mal_tensors)))
else:
    print("N_AUGMENT=0 and BALANCE_CLASSES=False — training on original train images only.")

## 9) Neural nets and DeepProbLog datasets

**What:** define four tiny CNNs (kind, shape, margin, calc) and wrap each row as a `malignant` query plus concept queries.

**Why:** DeepProbLog needs a neural predicate for each concept in the Prolog program. Concept queries teach the nets the CSV labels. The `malignant` query teaches the combination that the rules use. Networks are created later, once per grid-search trial, so each trial starts from a fresh initialization.


In [ ]:
class ConceptCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Linear(64 * 4 * 4, n_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # DeepProbLog may stack CPU tensors even if module is on CUDA — force match.
        if not torch.is_tensor(x):
            x = torch.as_tensor(x)
        x = x.to(next(self.parameters()).device, dtype=torch.float32)
        if x.dim() == 3:  # [C,H,W] -> [1,C,H,W]
            x = x.unsqueeze(0)
        return self.softmax(self.classifier(self.net(x).flatten(1)))


print("Networks defined. Fresh copies are created for each grid-search trial and for the final training run.")

from problog.logic import Term, Constant
from deepproblog.dataset import Dataset, DataLoader, QueryDataset
from deepproblog.query import Query


class TensorSource:
    def __init__(self, tensors, device=None):
        self.tensors = tensors
        self.device = device

    def __getitem__(self, item):
        t = self.tensors[int(item[0])]
        if self.device is not None:
            t = t.to(self.device)
        return t


class BreastQueryDataset(Dataset):
    def __init__(self, split_name, df):
        self.split_name = split_name
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def _tensor_term(self, i):
        return Term("tensor", Term(self.split_name, Constant(i)))

    def to_query(self, i):
        row = self.df.iloc[i]
        return Query(Term("malignant", self._tensor_term(i)), p=float(row["malignant"]))

    def concept_queries(self):
        qs = []
        for i, row in self.df.iterrows():
            img = self._tensor_term(i)
            qs.append(Query(Term("shape", img, Term(row["shape_concept"])), p=1.0))
            qs.append(Query(Term("margin", img, Term(row["margin_concept"])), p=1.0))
            qs.append(Query(Term("calc", img, Term(row["calc_concept"])), p=1.0))
            qs.append(Query(Term("lesion_kind", img, Term(row["kind_concept"])), p=1.0))
        return qs


print("Dataset classes ready. Query datasets are built inside the training function so grid search can rebuild them.")


## 10) Grid search, then train DeepProbLog

**What:** write `breast_diagnosis.pl`, optionally try each `(learning rate, batch size)` for a few epochs, pick the pair with the best **validation macro F1**, then train the final model for `EPOCHS`.

**Why:** the default `lr=1e-3` and `batch=4` were a first guess. A small grid on validation F1 chooses those two knobs from data instead of from habit. Search uses fewer epochs so it can finish on a T4; the final run still uses the full `EPOCHS`.

Mass evidence is used only when `lesion_kind` is mass. Suspicious calc is used only when `lesion_kind` is calcification.


In [ ]:
# import time
# from itertools import product
# from tqdm.auto import tqdm

# from deepproblog.engines import ExactEngine
# from deepproblog.model import Model
# from deepproblog.network import Network
# from deepproblog.train import train_model

# # Always write the notebook program. Do not reuse a stale Kaggle .pl from a previous run.
# program_path = OUTPUT_DIR / "breast_diagnosis.pl"
# program_path.write_text(PROGRAM_FALLBACK + "\n")
# print("Using program:", program_path.resolve())
# print(program_path.read_text())


# def _quick_prob(result):
#     mapping = result.result if hasattr(result, "result") else {}
#     if not mapping:
#         return 0.0
#     vals = []
#     for v in mapping.values():
#         if hasattr(v, "detach"):
#             vals.append(float(v.detach().cpu().item() if v.numel() == 1 else v.detach().cpu().mean()))
#         else:
#             vals.append(float(v))
#     return float(sum(vals) / len(vals)) if vals else 0.0


# def validation_macro_f1(trained_model, df):
#     """Score one trained model on the original validation images."""
#     y_true, y_pred = [], []
#     for i in range(len(df)):
#         q = Query(Term("malignant", Term("tensor", Term("val", Constant(i)))))
#         solved = trained_model.solve([q])
#         p = _quick_prob(solved[0]) if solved else 0.0
#         y_true.append(int(df.iloc[i]["malignant"]))
#         y_pred.append(int(p >= 0.5))
#     y_true, y_pred = np.array(y_true), np.array(y_pred)
#     tp = int(((y_pred == 1) & (y_true == 1)).sum())
#     tn = int(((y_pred == 0) & (y_true == 0)).sum())
#     fp = int(((y_pred == 1) & (y_true == 0)).sum())
#     fn = int(((y_pred == 0) & (y_true == 1)).sum())
#     f1_mal = 0.0 if (tp + fp) == 0 or (tp + fn) == 0 else (2 * tp / (2 * tp + fp + fn))
#     f1_ben = 0.0 if (tn + fn) == 0 or (tn + fp) == 0 else (2 * tn / (2 * tn + fn + fp))
#     return 0.5 * (f1_mal + f1_ben), {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "f1_malignant": f1_mal, "f1_benign": f1_ben}


# def train_deepproblog(lr, batch_size, epochs, log_label="train"):
#     """Fresh nets + DeepProbLog model. Returns (model, train_set)."""
#     shape_net = ConceptCNN(3).to(DEVICE)
#     margin_net = ConceptCNN(4).to(DEVICE)
#     calc_net = ConceptCNN(3).to(DEVICE)
#     kind_net = ConceptCNN(2).to(DEVICE)

#     train_set_local = BreastQueryDataset("train", df_train)
#     train_queries_local = QueryDataset(train_set_local.to_queries() + train_set_local.concept_queries())

#     shape_w = Network(shape_net, "shape_net", batching=True)
#     margin_w = Network(margin_net, "margin_net", batching=True)
#     calc_w = Network(calc_net, "calc_net", batching=True)
#     kind_w = Network(kind_net, "kind_net", batching=True)
#     shape_w.optimizer = torch.optim.Adam(shape_net.parameters(), lr=lr)
#     margin_w.optimizer = torch.optim.Adam(margin_net.parameters(), lr=lr)
#     calc_w.optimizer = torch.optim.Adam(calc_net.parameters(), lr=lr)
#     kind_w.optimizer = torch.optim.Adam(kind_net.parameters(), lr=lr)

#     trained = Model(str(program_path), [shape_w, margin_w, calc_w, kind_w])
#     trained.set_engine(ExactEngine(trained), cache=False)
#     trained.add_tensor_source("train", TensorSource(train_tensors, device=DEVICE))
#     trained.add_tensor_source("val", TensorSource(val_tensors, device=DEVICE))
#     trained.add_tensor_source("test", TensorSource(test_tensors, device=DEVICE))

#     loader = DataLoader(train_queries_local, batch_size, shuffle=True, seed=RANDOM_SEED)
#     n_batches = len(loader)
#     # Print progress ~50 times per epoch instead of once at the very end,
#     # so a long run shows visible movement instead of looking frozen.
#     log_iter = max(1, n_batches // 50)
#     print(f"[{log_label}] epochs={epochs} batch={batch_size} lr={lr} "
#           f"queries={len(train_queries_local)} batches/epoch={n_batches} "
#           f"(printing every {log_iter} batches)")
#     t0 = time.perf_counter()
#     train_model(trained, loader, epochs, log_iter=log_iter)
#     elapsed = time.perf_counter() - t0
#     print(f"[{log_label}] finished in {elapsed/60:.1f} min "
#           f"({elapsed / (epochs * n_batches):.2f} s/batch avg)")
#     return trained, train_set_local, shape_net, margin_net, calc_net, kind_net


# best_params = {"lr": LR, "batch_size": BATCH_SIZE}
# grid_rows = []

# if RUN_GRID_SEARCH:
#     combos = list(product(GRID_LEARNING_RATES, GRID_BATCH_SIZES))
#     print(f"Grid search on validation macro F1: {len(combos)} combos x {SEARCH_EPOCHS} epoch(s) each.")
#     print("(short runs; final training below still uses the full EPOCHS)")
#     t_grid_start = time.perf_counter()
#     for combo_i, (lr_i, bs_i) in enumerate(combos, start=1):
#         label = f"grid {combo_i}/{len(combos)} lr={lr_i} batch={bs_i}"
#         t_trial_start = time.perf_counter()
#         try:
#             trial_model, _, *_ = train_deepproblog(lr_i, bs_i, SEARCH_EPOCHS, log_label=label)
#             f1_macro, detail = validation_macro_f1(trial_model, df_val)
#             row = {"lr": lr_i, "batch_size": bs_i, "search_epochs": SEARCH_EPOCHS, "val_f1_macro": f1_macro, **detail, "ok": True}
#             print(f"  -> val_f1_macro={f1_macro:.4f}")
#         except Exception as exc:
#             row = {"lr": lr_i, "batch_size": bs_i, "search_epochs": SEARCH_EPOCHS, "val_f1_macro": -1.0, "ok": False, "error": f"{type(exc).__name__}: {str(exc)[:180]}"}
#             print("  -> failed:", row["error"])
#         grid_rows.append(row)
#         if DEVICE.type == "cuda":
#             torch.cuda.empty_cache()

#         trial_elapsed = time.perf_counter() - t_trial_start
#         total_elapsed = time.perf_counter() - t_grid_start
#         avg_per_combo = total_elapsed / combo_i
#         remaining = avg_per_combo * (len(combos) - combo_i)
#         print(f"  combo {combo_i}/{len(combos)} took {trial_elapsed/60:.1f} min | "
#               f"elapsed so far {total_elapsed/60:.1f} min | "
#               f"estimated remaining ~{remaining/60:.1f} min")
#     grid_df = pd.DataFrame(grid_rows)
#     display(grid_df.sort_values("val_f1_macro", ascending=False))
#     ok_rows = [r for r in grid_rows if r.get("ok")]
#     if ok_rows:
#         winner = max(ok_rows, key=lambda r: r["val_f1_macro"])
#         best_params = {"lr": winner["lr"], "batch_size": int(winner["batch_size"])}
#         print("Best grid pair:", best_params, "val_f1_macro=", winner["val_f1_macro"])
#     else:
#         print("Grid search had no successful run. Using config LR / BATCH_SIZE.")
#     (OUTPUT_DIR / "hyperparam_grid.json").write_text(json.dumps({"best": best_params, "rows": grid_rows}, indent=2, default=str))
#     print("Saved", OUTPUT_DIR / "hyperparam_grid.json")
#     print(f"Total grid search time: {(time.perf_counter() - t_grid_start)/60:.1f} min")
# else:
#     print("RUN_GRID_SEARCH=False — using config LR and BATCH_SIZE.")

# LR = float(best_params["lr"])
# BATCH_SIZE = int(best_params["batch_size"])
# print(f"Final training with lr={LR} batch={BATCH_SIZE} epochs={EPOCHS}")
# t_final_start = time.perf_counter()
# model, train_set, shape_net, margin_net, calc_net, kind_net = train_deepproblog(LR, BATCH_SIZE, EPOCHS, log_label="final")
# print(f"Final training total time: {(time.perf_counter() - t_final_start)/60:.1f} min")
# val_set = BreastQueryDataset("val", df_val)
# test_set = BreastQueryDataset("test", df_test)
# print("Training finished.")

In [ ]:
import time
import json
from pathlib import Path

import torch
from deepproblog.engines import ExactEngine
from deepproblog.model import Model
from deepproblog.network import Network
from deepproblog.train import train_model

program_path = OUTPUT_DIR / "breast_diagnosis.pl"
program_path.write_text(PROGRAM_FALLBACK + "\n")
print("Using program:", program_path.resolve())
print(program_path.read_text())

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "final_model_checkpoint.pt"

# We already ran the 6-combo grid search (took 62.8 min) and it picked
# lr=0.0003, batch_size=2 with val_f1_macro=0.5315. Skip redoing that search
# to avoid losing another ~hour if the session drops again. If you *want* to
# rerun the full search, flip this back to True (uses your original cell logic).
RUN_GRID_SEARCH_NOW = False
best_params = {"lr": 0.0003, "batch_size": 2}

grid_json_path = OUTPUT_DIR / "hyperparam_grid.json"
if grid_json_path.is_file():
    saved = json.loads(grid_json_path.read_text())
    best_params = saved.get("best", best_params)
    print("Loaded previous grid search result:", best_params)
else:
    print("No saved grid search file found on this fresh session "
          "— using the hardcoded winner from the last run:", best_params)


def _quick_prob(result):
    mapping = result.result if hasattr(result, "result") else {}
    if not mapping:
        return 0.0
    vals = []
    for v in mapping.values():
        if hasattr(v, "detach"):
            vals.append(float(v.detach().cpu().item() if v.numel() == 1 else v.detach().cpu().mean()))
        else:
            vals.append(float(v))
    return float(sum(vals) / len(vals)) if vals else 0.0


def validation_macro_f1(trained_model, df):
    """Score one trained model on the original validation images."""
    y_true, y_pred = [], []
    for i in range(len(df)):
        q = Query(Term("malignant", Term("tensor", Term("val", Constant(i)))))
        solved = trained_model.solve([q])
        p = _quick_prob(solved[0]) if solved else 0.0
        y_true.append(int(df.iloc[i]["malignant"]))
        y_pred.append(int(p >= 0.5))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    f1_mal = 0.0 if (tp + fp) == 0 or (tp + fn) == 0 else (2 * tp / (2 * tp + fp + fn))
    f1_ben = 0.0 if (tn + fn) == 0 or (tn + fp) == 0 else (2 * tn / (2 * tn + fn + fp))
    return 0.5 * (f1_mal + f1_ben), {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "f1_malignant": f1_mal, "f1_benign": f1_ben}


def build_fresh_model(lr):
    shape_net = ConceptCNN(3).to(DEVICE)
    margin_net = ConceptCNN(4).to(DEVICE)
    calc_net = ConceptCNN(3).to(DEVICE)
    kind_net = ConceptCNN(2).to(DEVICE)

    shape_w = Network(shape_net, "shape_net", batching=True)
    margin_w = Network(margin_net, "margin_net", batching=True)
    calc_w = Network(calc_net, "calc_net", batching=True)
    kind_w = Network(kind_net, "kind_net", batching=True)
    shape_w.optimizer = torch.optim.Adam(shape_net.parameters(), lr=lr)
    margin_w.optimizer = torch.optim.Adam(margin_net.parameters(), lr=lr)
    calc_w.optimizer = torch.optim.Adam(calc_net.parameters(), lr=lr)
    kind_w.optimizer = torch.optim.Adam(kind_net.parameters(), lr=lr)

    trained = Model(str(program_path), [shape_w, margin_w, calc_w, kind_w])
    trained.set_engine(ExactEngine(trained), cache=False)
    trained.add_tensor_source("train", TensorSource(train_tensors, device=DEVICE))
    trained.add_tensor_source("val", TensorSource(val_tensors, device=DEVICE))
    trained.add_tensor_source("test", TensorSource(test_tensors, device=DEVICE))
    return trained, (shape_net, margin_net, calc_net, kind_net)


def save_checkpoint(epoch, nets, lr, batch_size, path=CHECKPOINT_PATH):
    shape_net, margin_net, calc_net, kind_net = nets
    torch.save({
        "epoch": epoch,
        "shape_net": shape_net.state_dict(),
        "margin_net": margin_net.state_dict(),
        "calc_net": calc_net.state_dict(),
        "kind_net": kind_net.state_dict(),
        "lr": lr,
        "batch_size": batch_size,
    }, path)
    print(f"  checkpoint saved (epoch {epoch}) -> {path}")


def load_checkpoint_if_any(nets, path=CHECKPOINT_PATH):
    if not path.is_file():
        return 0
    shape_net, margin_net, calc_net, kind_net = nets
    ckpt = torch.load(path, map_location=DEVICE)
    shape_net.load_state_dict(ckpt["shape_net"])
    margin_net.load_state_dict(ckpt["margin_net"])
    calc_net.load_state_dict(ckpt["calc_net"])
    kind_net.load_state_dict(ckpt["kind_net"])
    print(f"Found checkpoint from a previous run: resuming after epoch {ckpt['epoch']} "
          f"(lr={ckpt['lr']}, batch={ckpt['batch_size']})")
    return ckpt["epoch"]


if RUN_GRID_SEARCH_NOW:
    raise RuntimeError(
        "RUN_GRID_SEARCH_NOW=True — paste your original grid-search loop here "
        "if you really want to redo the 6-combo search."
    )

LR = float(best_params["lr"])
BATCH_SIZE = int(best_params["batch_size"])
print(f"Final training with lr={LR} batch={BATCH_SIZE} epochs={EPOCHS}")

trained, nets = build_fresh_model(LR)
start_epoch = load_checkpoint_if_any(nets, CHECKPOINT_PATH)

train_set = BreastQueryDataset("train", df_train)
train_queries = QueryDataset(train_set.to_queries() + train_set.concept_queries())
loader = DataLoader(train_queries, BATCH_SIZE, shuffle=True, seed=RANDOM_SEED)
n_batches = len(loader)
log_iter = max(1, n_batches // 50)
print(f"queries={len(train_queries)} batches/epoch={n_batches} (printing every {log_iter} batches)")

t_final_start = time.perf_counter()
for epoch in range(start_epoch, EPOCHS):
    print(f"\n=== Epoch {epoch + 1}/{EPOCHS} ===")
    t_ep = time.perf_counter()
    train_model(trained, loader, 1, log_iter=log_iter)  # one epoch per call, so we can checkpoint between epochs
    print(f"  epoch {epoch + 1} time: {time.perf_counter() - t_ep:.1f}s")
    save_checkpoint(epoch + 1, nets, LR, BATCH_SIZE, CHECKPOINT_PATH)

print(f"Final training total time: {(time.perf_counter() - t_final_start) / 60:.1f} min")

model = trained
shape_net, margin_net, calc_net, kind_net = nets
val_set = BreastQueryDataset("val", df_val)
test_set = BreastQueryDataset("test", df_test)
print("Training finished.")

## 11) Metrics, threshold selection, reasoning traces, and predicted test images

**What:** score every validation and test image once (`P(malignant)`), then:

1. **Pick the decision threshold on validation** using `THRESHOLD_MODE` (default: the lowest threshold that reaches `TARGET_SENSITIVITY` on validation). Test is scored with that same fixed value. A table compares the 0.5 / Youden / macro-F1 / target-sensitivity rules so the choice is transparent.
2. Report accuracy, precision, sensitivity, specificity, F1 (malignant, benign, macro) and **ROC-AUC** at both `0.5` and the selected threshold. ROC-AUC is threshold-free, so it tells you whether the ranking improved, not just the cut-off.
3. Draw the ROC curves, the validation sensitivity/specificity sweep, and a histogram of `P(malignant)` per true class.
4. Print **reasoning traces**: for each example, every concept probability, the probability of each rule body, the strongest rule, and the decision. Includes one false negative and one false positive from test.
5. Draw validation and test images with predictions. Green title = match; red title = mismatch.

**Why:** with ~40% malignant, 0.5 is not a neutral cut-off; lowering it trades specificity for sensitivity, which is the right trade for screening. The traces are the DeepProbLog explanation: the final probability is a deterministic function of the concept probabilities through the three rules, so every decision can be read back to *which* concept the network saw.


In [ ]:
def extract_prob(result):
    if result is None:
        return 0.0
    mapping = result.result if hasattr(result, "result") else (result if isinstance(result, dict) else {})
    if not mapping:
        return 0.0
    vals = []
    for v in mapping.values():
        if hasattr(v, "detach"):
            vals.append(float(v.detach().cpu().item() if v.numel() == 1 else v.detach().cpu().mean()))
        else:
            vals.append(float(v))
    return float(sum(vals) / len(vals))


def malignant_probability(split_name, i):
    q = Query(Term("malignant", Term("tensor", Term(split_name, Constant(i)))))
    solved = model.solve([q])
    return extract_prob(solved[0]) if solved else 0.0


def _f1(tp, fp, fn):
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    if prec + rec == 0:
        return 0.0
    return 2.0 * prec * rec / (prec + rec)


def collect_probs(name, dataset, df):
    """Run the malignant(X) query once per image. Expensive, so do it once per split."""
    y_true, y_prob = [], []
    for i in tqdm(range(len(dataset)), desc=f"Scoring {name}", unit="img"):
        y_true.append(int(df.iloc[i]["malignant"]))
        y_prob.append(malignant_probability(name, i))
    return np.array(y_true), np.array(y_prob)


def metrics_at(name, y_true, y_prob, thr):
    """All metrics for one split at one decision threshold."""
    y_pred = (y_prob >= thr).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    f1_mal = _f1(tp, fp, fn)
    f1_ben = _f1(tn, fn, fp)
    try:
        auc = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else float("nan")
    except Exception:
        auc = float("nan")
    return {
        "split": name, "threshold": float(thr), "n": int(len(y_true)),
        "accuracy": (tp + tn) / max(len(y_true), 1),
        "precision": precision,
        "sensitivity": tp / max(tp + fn, 1),
        "specificity": tn / max(tn + fp, 1),
        "f1_malignant": f1_mal,
        "f1_benign": f1_ben,
        "f1_macro": 0.5 * (f1_mal + f1_ben),
        "roc_auc": auc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "mean_p": float(y_prob.mean()),
        "pct_pred_malignant": float(y_pred.mean()),
    }


def choose_threshold(y_true, y_prob, mode, target_sens=0.80, fixed=0.5):
    """Pick the decision threshold on VALIDATION probabilities only.

    Returns (threshold, sweep_dataframe). The sweep is kept so the choice is
    visible: every candidate threshold with its sensitivity/specificity/F1.
    """
    grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
    sweep = pd.DataFrame([metrics_at("val", y_true, y_prob, t) for t in grid])
    sweep["youden_j"] = sweep["sensitivity"] + sweep["specificity"] - 1.0
    if mode == "fixed":
        return float(fixed), sweep
    if mode == "youden":
        return float(sweep.loc[sweep["youden_j"].idxmax(), "threshold"]), sweep
    if mode == "f1_macro":
        return float(sweep.loc[sweep["f1_macro"].idxmax(), "threshold"]), sweep
    if mode == "target_sensitivity":
        ok = sweep[sweep["sensitivity"] >= target_sens]
        if len(ok) == 0:
            print(f"No threshold reaches sensitivity >= {target_sens}; falling back to Youden's J.")
            return float(sweep.loc[sweep["youden_j"].idxmax(), "threshold"]), sweep
        # Among thresholds that hit the recall target, take the one with the best specificity.
        return float(ok.loc[ok["specificity"].idxmax(), "threshold"]), sweep
    raise ValueError(f"Unknown THRESHOLD_MODE={mode!r}")


from sklearn.metrics import roc_auc_score, roc_curve

y_val_true, val_prob = collect_probs("val", val_set, df_val)
y_test_true, test_prob = collect_probs("test", test_set, df_test)

SELECTED_THRESHOLD, val_sweep = choose_threshold(
    y_val_true, val_prob, THRESHOLD_MODE, target_sens=TARGET_SENSITIVITY, fixed=DECISION_THRESHOLD
)
print(f"\nTHRESHOLD_MODE={THRESHOLD_MODE!r} -> selected threshold on validation = {SELECTED_THRESHOLD:.2f}")
print("(chosen on validation only; test uses the same value unchanged)\n")

# How the choice would differ under each rule — useful when writing the paper.
alt = {
    "fixed 0.5": 0.5,
    "youden": float(val_sweep.loc[val_sweep["youden_j"].idxmax(), "threshold"]),
    "f1_macro": float(val_sweep.loc[val_sweep["f1_macro"].idxmax(), "threshold"]),
    f"target_sensitivity>={TARGET_SENSITIVITY}": SELECTED_THRESHOLD if THRESHOLD_MODE == "target_sensitivity"
        else choose_threshold(y_val_true, val_prob, "target_sensitivity", TARGET_SENSITIVITY)[0],
}
rows = []
for rule, thr in alt.items():
    for split, yt, yp in (("val", y_val_true, val_prob), ("test", y_test_true, test_prob)):
        m = metrics_at(split, yt, yp, thr)
        rows.append({"rule": rule, **m})
compare_df = pd.DataFrame(rows)[["rule", "threshold", "split", "accuracy", "sensitivity", "specificity",
                                  "precision", "f1_malignant", "f1_benign", "f1_macro", "roc_auc", "tp", "tn", "fp", "fn"]]
print("Threshold rules compared (each threshold chosen on val, applied to test):")
display(compare_df.round(3))

# Final numbers: at 0.5 (reference) and at the selected threshold.
metrics_val_05 = metrics_at("val", y_val_true, val_prob, 0.5)
metrics_test_05 = metrics_at("test", y_test_true, test_prob, 0.5)
metrics_val = metrics_at("val", y_val_true, val_prob, SELECTED_THRESHOLD)
metrics_test = metrics_at("test", y_test_true, test_prob, SELECTED_THRESHOLD)
metrics_df = pd.DataFrame([metrics_val_05, metrics_test_05, metrics_val, metrics_test])
print("\nReference (0.5) vs selected threshold:")
display(metrics_df.round(3))
print(
    "If the model predicts only one class: specificity or sensitivity hits 0, "
    "f1_benign or f1_malignant hits 0, and f1_macro stays low even if accuracy looks ok. "
    "ROC-AUC does not depend on the threshold, so it tells you how good the ranking itself is."
)
if SMOKE_TEST:
    print("SMOKE_TEST=True: this is a tiny wiring test. Treat the numbers as a smoke-test check only.")

# Visual: ROC curves + validation sensitivity/specificity vs threshold.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for split, yt, yp in (("val", y_val_true, val_prob), ("test", y_test_true, test_prob)):
    if len(np.unique(yt)) == 2:
        fpr, tpr, _ = roc_curve(yt, yp)
        axes[0].plot(fpr, tpr, label=f"{split} (AUC={roc_auc_score(yt, yp):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set_xlabel("1 - specificity (FPR)"); axes[0].set_ylabel("sensitivity (TPR)"); axes[0].set_title("ROC")
axes[0].legend()
axes[1].plot(val_sweep["threshold"], val_sweep["sensitivity"], label="val sensitivity")
axes[1].plot(val_sweep["threshold"], val_sweep["specificity"], label="val specificity")
axes[1].plot(val_sweep["threshold"], val_sweep["f1_macro"], label="val macro F1")
axes[1].axvline(SELECTED_THRESHOLD, color="red", ls="--", label=f"selected = {SELECTED_THRESHOLD:.2f}")
axes[1].axvline(0.5, color="gray", ls=":", label="0.5")
axes[1].set_xlabel("threshold on P(malignant)"); axes[1].set_title("Validation sweep")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

# Histogram of P(malignant) by true class: shows whether the two classes separate at all.
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(test_prob[y_test_true == 0], bins=30, alpha=0.6, label="test benign")
ax.hist(test_prob[y_test_true == 1], bins=30, alpha=0.6, label="test malignant")
ax.axvline(SELECTED_THRESHOLD, color="red", ls="--", label=f"threshold {SELECTED_THRESHOLD:.2f}")
ax.set_xlabel("P(malignant)"); ax.set_ylabel("count"); ax.legend()
ax.set_title("Test P(malignant) by true class")
fig.tight_layout()
plt.show()


def inspect_case(split_name, df, i, thr=None):
    """Print the full reasoning trace for one image.

    This is the DeepProbLog explanation: the final P(malignant) is computed
    from the concept probabilities through the three rules in the .pl program,
    so you can see WHICH rule fired and WHICH concept drove the decision.
    """
    thr = SELECTED_THRESHOLD if thr is None else thr
    row = df.iloc[i]
    img = Term("tensor", Term(split_name, Constant(i)))
    queries = {
        "malignant": Query(Term("malignant", img)),
        "kind_mass": Query(Term("lesion_kind", img, Term("mass"))),
        "kind_calc": Query(Term("lesion_kind", img, Term("calcification"))),
        "shape_irregular": Query(Term("shape", img, Term("irregular"))),
        "margin_spiculated": Query(Term("margin", img, Term("spiculated"))),
        "calc_suspicious": Query(Term("calc", img, Term("suspicious"))),
    }
    p = {}
    for name, q in queries.items():
        solved = model.solve([q])
        p[name] = extract_prob(solved[0]) if solved else 0.0
    gold = "MALIGNANT" if int(row["malignant"]) == 1 else "BENIGN"
    pred = "MALIGNANT" if p["malignant"] >= thr else "BENIGN"
    verdict = "correct" if gold == pred else "WRONG"
    print(f"\n[{split_name} #{i}] patient={row['patient_id']} pathology={row['pathology']} kind={row['abnormality_kind']}")
    print(f"  gold: shape={row['shape_concept']} margin={row['margin_concept']} calc={row['calc_concept']} kind={row.get('kind_concept')}")
    if "resolved_path" in row.index:
        print(f"  image: {row['resolved_path']}")
    for name, v in p.items():
        print(f"  P({name}) = {v:.3f}")
    # Per-rule contribution: probability that each rule body is true (independence assumed by the program).
    r1 = p["kind_mass"] * p["shape_irregular"]
    r2 = p["kind_mass"] * p["margin_spiculated"]
    r3 = p["kind_calc"] * p["calc_suspicious"]
    top = max((("mass & irregular shape", r1), ("mass & spiculated margin", r2), ("calcification & suspicious calc", r3)),
              key=lambda kv: kv[1])
    print(f"  rule bodies: mass&irregular={r1:.3f}  mass&spiculated={r2:.3f}  calc&suspicious={r3:.3f}")
    print(f"  strongest rule: {top[0]}")
    print(f"  decision @ {thr:.2f}: {pred}  (gold {gold}) -> {verdict}")


print("Example reasoning traces (validation):")
for i in range(min(3, len(val_set))):
    inspect_case("val", df_val, i)

# Also show the traces the paper needs most: a false negative and a false positive on test.
y_test_pred = (test_prob >= SELECTED_THRESHOLD).astype(int)
fn_idx = np.where((y_test_pred == 0) & (y_test_true == 1))[0]
fp_idx = np.where((y_test_pred == 1) & (y_test_true == 0))[0]
if len(fn_idx):
    print("\nExample FALSE NEGATIVE on test (missed cancer — the case that matters most):")
    inspect_case("test", df_test, int(fn_idx[0]))
if len(fp_idx):
    print("\nExample FALSE POSITIVE on test:")
    inspect_case("test", df_test, int(fp_idx[0]))


def gold_concept_prob(split_name, i, predicate, label):
    q = Query(Term(predicate, Term("tensor", Term(split_name, Constant(i))), Term(label)))
    solved = model.solve([q])
    return extract_prob(solved[0]) if solved else 0.0


print("\nValidation concept check (mean P(gold label); >0.5 means the net usually agrees with the CSV):")
concept_check = {}
for pred, col in (
    ("lesion_kind", "kind_concept"),
    ("shape", "shape_concept"),
    ("margin", "margin_concept"),
    ("calc", "calc_concept"),
):
    probs = [gold_concept_prob("val", i, pred, df_val.iloc[i][col]) for i in range(len(df_val))]
    arr = np.array(probs)
    concept_check[pred] = {"mean_p_gold": float(arr.mean()), "agree_at_0.5": float((arr >= 0.5).mean())}
    print(f"  {pred:12s}  mean P(gold)={arr.mean():.3f}  agree@0.5={(arr >= 0.5).mean():.3f}")


def show_prediction_grid(title, df, tensors, y_prob, n=None, thr=None):
    thr = SELECTED_THRESHOLD if thr is None else thr
    n = min(int(n if n is not None else SHOW_N_IMAGES), len(df), len(tensors))
    y_pred = (np.asarray(y_prob) >= thr).astype(int)
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(13, 3.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        ax = axes[i]
        ax.imshow(tensor_to_numpy(tensors[i]), cmap="gray")
        row = df.iloc[i]
        gold = "MAL" if int(row["malignant"]) == 1 else "BEN"
        pred = "MAL" if int(y_pred[i]) == 1 else "BEN"
        ok = gold == pred
        ax.set_title(
            f"{row['patient_id']}  gold={gold} pred={pred}\nP={y_prob[i]:.2f}  {row['abnormality_kind']}",
            fontsize=8,
            color=("green" if ok else "red"),
        )
        ax.axis("off")
    for j in range(n, len(axes)):
        axes[j].axis("off")
    fig.suptitle(f"{title} (threshold {thr:.2f})")
    fig.tight_layout()
    plt.show()


print(f"\nPredicted test / validation images at threshold {SELECTED_THRESHOLD:.2f} (green = match, red = mismatch):")
show_prediction_grid("Validation images + predictions", df_val, val_tensors, val_prob)
show_prediction_grid("Test images + predictions", df_test, test_tensors, test_prob)

## 12) Save weights and metrics

**What:** store the model snapshot and a metrics JSON under `OUTPUT_DIR`.

**Why:** so you can reload this run later without training again.


In [ ]:
snapshot = OUTPUT_DIR / ("deepproblog_smoke.pth" if SMOKE_TEST else "deepproblog_model.pth")
model.save_state(str(snapshot))
payload = {
    "smoke_test": SMOKE_TEST,
    "epochs": EPOCHS,
    "image_size": IMAGE_SIZE,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "n_augment": N_AUGMENT,
    "balance_classes": BALANCE_CLASSES,
    "grid_search": RUN_GRID_SEARCH,
    "best_params": best_params,
    "threshold_mode": THRESHOLD_MODE,
    "target_sensitivity": TARGET_SENSITIVITY,
    "selected_threshold": SELECTED_THRESHOLD,
    "n_train": len(train_set),
    "n_val": len(val_set),
    "n_test": len(test_set),
    "metrics_val_at_0.5": metrics_val_05,
    "metrics_test_at_0.5": metrics_test_05,
    "metrics_val": metrics_val,
    "metrics_test": metrics_test,
    "concept_check_val": concept_check,
    "note": "Smoke-test numbers are a wiring check only. Rules are a computational program, not official BI-RADS.",
}
(OUTPUT_DIR / "deepproblog_metrics.json").write_text(json.dumps(payload, indent=2, default=float))
compare_df.to_csv(OUTPUT_DIR / "threshold_rules_comparison.csv", index=False)
val_sweep.to_csv(OUTPUT_DIR / "val_threshold_sweep.csv", index=False)
np.save(OUTPUT_DIR / "val_prob.npy", val_prob)
np.save(OUTPUT_DIR / "test_prob.npy", test_prob)
print("Saved:", snapshot)
print("Saved:", OUTPUT_DIR / "deepproblog_metrics.json")
print("Saved:", OUTPUT_DIR / "threshold_rules_comparison.csv", "and", OUTPUT_DIR / "val_threshold_sweep.csv")
print("Done. Single-notebook pipeline complete.")
